# Evaluation and Reliability Metrics for Agentic Systems

Agent quality is **multi-level**: task outcomes, latency, grounding, RAG-style context quality, multi-agent coordination, cost, and reference datasets. This notebook exercises helpers from `agentic_assistants.core.evaluation` with **synthetic** observations.

## Introduction: multi-level evaluation strategy

Operational metrics (success rate, p95 latency) catch regressions quickly. **Hallucination** and **RAG** proxies catch retrieval and grounding issues. **Multi-agent** metrics highlight coordination waste. **Resource** tracking ties reliability to cost. **Golden datasets** anchor behavior to expected outputs across match modes.

In [ ]:
try:
    from agentic_assistants.core.evaluation import (
        TaskSuccessRate,
        LatencyMetric,
        HallucinationRate,
        ContextRelevance,
        ContextSufficiency,
        AnswerRelevance,
        HandoffEfficiency,
        CommunicationEfficiency,
        ResourceFootprint,
        GoldenDataset,
        EvaluationMixin,
        EvaluationResult,
        MetricTracker,
    )
    EVAL_OK = True
except ImportError as e:
    EVAL_OK = False
    print("Import failed:", e)

## Task success rate and latency

`TaskSuccessRate` aggregates boolean outcomes; `LatencyMetric` accepts precomputed millisecond samples or `record(start_time)` from `time.perf_counter()`.

In [ ]:
if EVAL_OK:
    import time

    tsr = TaskSuccessRate()
    for ok in [True, True, False, True, True]:
        tsr.record(ok)
    print("Task success rate:", round(tsr.rate, 3), "n=", tsr.total)

    lat = LatencyMetric()
    for ms in [120.0, 95.0, 210.0, 140.0, 400.0, 130.0, 160.0, 180.0, 900.0, 150.0]:
        lat.record_value(ms)
    print("p50", lat.p50, "p95", lat.p95, "mean", round(lat.mean, 1))

## Hallucination rate tracking

Treat each response as **grounded** or not (from citations, entailment models, or human labels). `HallucinationRate.rate` is the fraction hallucinated.

In [ ]:
if EVAL_OK:
    hal = HallucinationRate()
    for grounded in [True, True, True, False, True]:
        hal.record(grounded)
    print("Hallucination rate:", round(hal.rate, 3))

## RAGAS-inspired metrics (rule-based proxies)

`ContextRelevance`, `ContextSufficiency`, and `AnswerRelevance` use token overlap proxies—**no LLM required**—useful for quick dashboards; swap for model-based scores in production if needed.

In [ ]:
if EVAL_OK:
    cr = ContextRelevance()
    cr.record("What is the capital of France?", "France is a country in Europe. Paris is its capital city.")
    cs = ContextSufficiency()
    cs.record("Paris", "France is a country in Europe. Paris is its capital city.")
    ar = AnswerRelevance()
    ar.record("capital of France?", "The capital of France is Paris.")
    print("context relevance mean", round(cr.mean, 3))
    print("context sufficiency mean", round(cs.mean, 3))
    print("answer relevance mean", round(ar.mean, 3))

## Multi-agent metrics: handoff and communication

`HandoffEfficiency` tracks successful handoffs plus oscillations and deadlocks. `CommunicationEfficiency` tracks whether messages are **useful** and token volume.

In [ ]:
if EVAL_OK:
    ho = HandoffEfficiency()
    ho.record_handoff(True)
    ho.record_handoff(True)
    ho.record_handoff(False)
    ho.record_oscillation()
    print("handoff success", round(ho.success_rate, 3), "oscillation rate", round(ho.oscillation_rate, 3))

    ce = CommunicationEfficiency()
    ce.record_message(useful=True, token_count=50)
    ce.record_message(useful=False, token_count=200)
    ce.record_message(useful=True, token_count=40)
    print("usefulness", round(ce.usefulness_rate, 3), "avg tokens/msg", round(ce.avg_tokens_per_message, 1))

## Resource footprint (cost tracking)

`ResourceFootprint` accumulates API calls, tokens, and USD cost, then normalizes per successful **resolution**.

In [ ]:
if EVAL_OK:
    rf = ResourceFootprint()
    rf.record_call(tokens=1200, cost_usd=0.002)
    rf.record_call(tokens=800, cost_usd=0.0015)
    rf.record_resolution()
    print("cost per resolution", rf.cost_per_resolution)
    print("tokens per resolution", rf.tokens_per_resolution)

## Golden dataset: multiple match modes

Load JSON/JSONL references and compare predictions with `exact`, `fuzzy`, `substring`, or simplified `bleu` scoring.

In [ ]:
import json
import tempfile
from pathlib import Path

if EVAL_OK:
    golden_items = [
        {"expected": "Paris", "actual": "Paris"},
        {"expected": "The answer is 42.", "actual": "answer is 42"},
        {"expected": "yes", "actual": "no"},
    ]
    tmp = Path(tempfile.mkdtemp()) / "golden.json"
    tmp.write_text(json.dumps(golden_items), encoding="utf-8")
    ds = GoldenDataset(tmp)
    preds = ["Paris", "The answer is 42.", "yes"]
    for mode in ("exact", "fuzzy", "substring", "bleu"):
        r = ds.evaluate(preds, mode=mode, fuzzy_threshold=0.75)
        print(mode, r)
    # In-item predictions
    r2 = GoldenDataset(tmp).evaluate([], prediction_key="actual", mode="exact")
    print("from items actual key", r2)

## `EvaluationMixin` integration

Subclass and implement `_collect_eval_data()`; call `compute_reliability_metrics()` for an `EvaluationResult` summary.

In [ ]:
if EVAL_OK:
    class SyntheticAgentRun(EvaluationMixin):
        def _collect_eval_data(self):
            return {
                "successes": [True, True, False],
                "latencies": [100.0, 150.0, 200.0],
                "groundedness": [True, False, True],
                "handoffs": [True, True],
                "oscillations": 1,
                "deadlocks": 0,
                "context_pairs": [{"query": "q1", "context": "q1 context here"}],
                "sufficiency_pairs": [{"answer": "a1", "context": "a1 and more"}],
                "answer_pairs": [{"query": "q2", "answer": "q2 answer"}],
                "messages": [{"useful": True, "tokens": 30}, {"useful": False, "tokens": 100}],
                "api_calls": [{"tokens": 500, "cost_usd": 0.001}],
                "resolutions": 2,
                "custom_note": "synthetic batch",
            }

    agent = SyntheticAgentRun()
    result = agent.compute_reliability_metrics()
    assert isinstance(result, EvaluationResult)
    print(json.dumps(result.summary(), indent=2, default=str))
    print("custom", result.custom_metrics)

## `MetricTracker` (lightweight accumulator)

For ad-hoc named metrics without building a full `EvaluationResult`.

In [ ]:
if EVAL_OK:
    mt = MetricTracker()
    mt.record_many({"reward": 0.8, "tool_errors": 0.0})
    mt.record_many({"reward": 0.9, "tool_errors": 1.0})
    print(mt.summary())

## Conclusion

- Combine **outcome + latency + cost** for SLO-style monitoring.
- Layer **grounding** and **RAG proxies** when retrieval is in the loop.
- Track **handoffs** and **message usefulness** as systems grow multi-agent.
- Use **GoldenDataset** with the right **match mode** for your tolerance of paraphrase.
- **`EvaluationMixin`** centralizes aggregation when your agent already exposes raw episode logs.

Treat these metrics as **diagnostic signals**—they narrow down where to invest in prompts, tools, or architecture.